# Arquitetura Transformers

## Contextualização

Antes de 2017, o processamento de linguagem natural (NLP) era dominado por Redes Neurais Recorrentes (RNNs) e LSTMs. Essas arquiteturas processam os dados sequencialmente (palavra por palavra). Dois problemas são associados a esse tipo de implementação:

1.  **Impossibilidade de Paralelização:** Como o estado da palavra atual dependia da anterior, não era possível usar todo o poder das GPUs para processar uma frase inteira de uma vez.
2.  **Esquecimento de Longa Distância:** Em frases muito longas, a relação entre a primeira e a última palavra frequentemente se perdia, mesmo com mecanismos como LSTMs (problemática vista no laboratório de RNN e LSTM).

A arquitetura Transformer, introduzida no paper *"[Attention Is All You Need](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf?src_trk=em66c5166b7bad47.12959706424423622)"*, resolveu isso descartando a recorrência e baseando-se inteiramente no mecanismo de Atenção. Isso permite que o modelo olhe para a frase inteira simultaneamente.

Para desmistificar o funcionamento interno da arquitetura Transformer, uma das melhores referências visuais e o guia [The Illustrated Transformer by Jay Alammar](https://jalammar.github.io/illustrated-transformer/). O Autor comparar o modelo a uma pilha de blocos de processamento, onde os dados fluem de baixo para cima, ganhando representações cada vez mais ricas a cada camada. Recomenda-se a leitura do artigo para um melhor entendimento de como as arquiteturas Transformers funcionam.

É importante notar uma distinção crucial entre a aplicação completa e o que vai ser feito nesse laboratório. O Transformer original (proposto no artigo e exemplificado no material) possui duas partes, um codificador (que processa a entrada) e um decodificador (que gera a saída). O trabalho que será feito nesse laboratório para a tarefa de classificação de texto será usado apenas a parte do codificador. Neste laboratório, será feito uma arquitetura de uma forma mais manual, e numa segunda parte, será utilizando uma arquitetura mais robusta chamada BERT, treinada usando a técnica de *fine-tunning*.

Alguns pontos de implementação são úteis para destacar neste laboratório:

* **Positional Encoding**: Como foi removido a recorrência, o modelo não sabe nativamente a ordem das palavras. "O cão mordeu o homem" e "O homem mordeu o cão" pareceriam iguais para uma rede sem noção de sequência. Para resolver isso, será somado vetores matemáticos que marcam a posição de cada palavra na frase.
* **Self-attention**: Permite que cada palavra olhe para todas as outras palavras da frase simultaneamente para entender o contexto. Por exemplo, na frase "o banco estava fechado", o mecanismo de atenção ajuda o modelo a associar a palavra "banco" mais fortemente com "fechado" e "dinheiro" do que com "parque" ou "assento".

Em etapas anteriores, foi feita a construção de um modelo baseado em RNNs simples, onde foi possível observar na prática o problema do *Vanishing Gradient* (Desaparecimento do Gradiente), que impedia o aprendizado de sequências longas. A solução imediata foi a implementação de LSTMs (*Long-Short-Term Memory*), que mitiga esse problema através de portas de controle (*gates*) para reter informações importantes por mais tempo.

No entanto, embora as LSTMs sejam poderosas, elas ainda sofrem com a natureza sequencial do processamento, o que limita a capacidade do modelo de entender contextos globais complexos e torna o treinamento lento (em situações de aprendizagem mais complexas).

Agora, dando o próximo passo, será utilizado Transformers. Ao contrário das LSTMs, que leem o texto como uma fita continua, o Transformer consegue enxergar toda a sentença de uma vez. Isso não apenas acelera o treinamento através de paralelização, mas, crucialmente, permite capturar nuances semânticas e relações entre palavras distantes com uma precisão melhor da que foi feita com o modelo de LSTM em laboratórios passados.

## Conjunto de Dados (Recapitulação)

Para a validação deste modelo, reutilizaremos o conjunto de dados IMDB, cuja análise exploratória, balanceamento de classes e nuvens de palavras já foram detalhadamente discutidos no laboratório sobre RNN e LSTM.

Apenas como lembrete técnico para a configuração da arquitetura atual, o conjunto de dados trata-se de um problema de classificação binária (Positivo/Negativo). Para adequar os dados à entrada do Transformer e manter uma base de comparação justa com nossos modelos prévios (RNN/LSTM), foi mantido o vocabulário restrito às 20.000 palavras mais frequentes e foi padronizado o comprimento das sequências (*padding/truncating*) para 200 tokens. Isso garante que o foco do experimento seja a eficiência da arquitetura de atenção, isolando variáveis de pré-processamento.

In [ ]:
#!pip install tensorflow keras-preprocessing

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import torch.nn as nn
import torch

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb 

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [ ]:
# verificando compatibilidade com GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Recurso Gráfico: {device}")

In [ ]:
# configuração de hiperparâmetros
VOCAB_SIZE = 10000  
MAXLEN = 500
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 32
BATCH_SIZE = 32
EPOCHS = 6
LEARNING_RATE = 1e-3

In [ ]:
def prepare_data(vocab_size=10000, maxlen=500, batch_size=32):
    # prepara dataloaders de treino e validação do conjunto de dados IMDB

    # carrega o IMDB mantendo apenas as palavras mais frequentes do vocabulário
    (x_train, y_train), (x_val, y_val) = imdb.load_data(num_words=vocab_size)

    # padroniza todas as sequências para o mesmo comprimento
    x_train = pad_sequences(x_train, maxlen=maxlen)
    x_val = pad_sequences(x_val, maxlen=maxlen)
    
    # converte dados e rótulos para tensores PyTorch
    train_data = TensorDataset(torch.tensor(x_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.long))
    val_data = TensorDataset(torch.tensor(x_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.long))

    # cria dataloaders para iteração eficiente em batches
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    
    return train_loader, val_loader

In [ ]:
# Preparação dos dados
train_loader, val_loader = prepare_data(VOCAB_SIZE, MAXLEN, BATCH_SIZE)

## Modelo Transformer

O diagrama abaixo detalha o fluxo de tensores da arquitetura construída para resolver o problema de avaliação de resenhas com os dados do IMDB.

<figure style="text-align: center;">
  <img src="assets/figura-3.png" alt="Figura 3" width="800">
</figure>

**Figura 1** – Diagrama da arquitetura transformer.

Do ponto de vista do estado da arte, é fundamental estabelecer que esta é uma versão reduzida da implementação original proposta no artigo *"[Attention Is All You Need](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf?src_trk=em66c5166b7bad47.12959706424423622)"*. 

Desse modo, a arquitetura foi podada e adaptada, mantendo apenas os seguintes componentes:

(I). **Pré-Processamento:** Em substituição às funções matemáticas estáticas (seno e cosseno) propostas no artigo original para a codificação espacial, optamos por instanciar embeddings posicionais aprendidos. Estes pesos são atualizados via retropropagação e somados aos embeddings dos tokens de entrada ($X_1 \dots X_n$), garantindo que a rede paralela não perca a noção de ordem temporal.

(II). **Codificador (Encoder):** Atua exclusivamente no processamento da entrada. O mecanismo de Autoatenção (*Self-Attention*) permite que a rede correlacione cada *token* com todos os outros da mesma sentença simultaneamente. Para contornar gargalos de treinamento, atalhos residuais propagam os tensores inalterados através das camadas, mitigando o desaparecimento do gradiente.

(III). **Saída do Modelo:** Como não há geração de texto, a metade decodificadora do Transformer foi completamente descartada. Em seu lugar, foi acoplado uma operação temporal de agregação global (*average pooling*), que comprime o processamento das 200 posições da sequência em um vetor de contexto único e denso. Esse vetor alimenta uma camada linear final que emite a predição.

In [ ]:
class TransformerBlock(nn.Module):
    # bloco codificador transformer com self-attention e feed-forward
    
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerBlock, self).__init__()
        
        # camada de atenção sobre a sequência de embeddings
        self.att = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        
        # rede feed forward aplicada a cada posição da sequência
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        
        # normalização e regularização para estabilidade
        self.layernorm1 = nn.LayerNorm(embed_dim)
        self.layernorm2 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        # camada de atenção
        attn_output, _ = self.att(x, x, x)  # Self-attention contextual
        attn_output = self.dropout1(attn_output)
        # Conexão residual
        out1 = self.layernorm1(x + attn_output) # Conexão residual
        
        # camada Feed Forward
        ffn_output = self.ffn(out1)  # Transformação não linear por token
        ffn_output = self.dropout2(ffn_output)
        
        return self.layernorm2(out1 + ffn_output) # Saída normalizada

In [ ]:
class TokenAndPositionEmbedding(nn.Module):
    # soma embeddings de tokens e posições para entrada do transformer

    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim) # Embedding das palavras
        self.pos_emb = nn.Embedding(maxlen, embed_dim) # Embedding das posições
        self.maxlen = maxlen

    def forward(self, x):
        # x: (batch_size, seq_len)
        positions = torch.arange(0, self.maxlen, device=x.device) # Índices posicionais
        token_embeddings = self.token_emb(x) # (batch, seq_len, embed_dim)
        position_embeddings = self.pos_emb(positions) # (seq_len, embed_dim)
        
        return token_embeddings + position_embeddings # Soma token + posição


In [ ]:
class IMDBTransformer(nn.Module):
    # transformer simplificado para classificação de sentimentos no dataset do IMDB

    def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, num_classes=2):
        super(IMDBTransformer, self).__init__()
        self.embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim) # Tokens + posições
        self.transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim) # Bloco Transformer

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1) # Pooling global na dimensão temporal
        self.dropout = nn.Dropout(0.1) # Regularização
        self.fc1 = nn.Linear(embed_dim, 20) # Projeção intermediária
        self.relu = nn.ReLU() # Não linearidade
        self.fc2 = nn.Linear(20, num_classes) # Camada de saída

    def forward(self, x):
        # x: (batch_size, seq_len)
        x = self.embedding_layer(x) # (batch, seq_len, embed_dim)
        x = self.transformer_block(x) # (batch, seq_len, embed_dim)

        x = x.permute(0, 2, 1) # (batch, embed_dim, seq_len)
        x = self.global_avg_pool(x) # (batch, embed_dim, 1)
        x = x.squeeze(-1) # (batch, embed_dim)

        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x) # (batch, num_classes)

        return x

In [ ]:
# Inicializando o modelo
model = IMDBTransformer(VOCAB_SIZE, MAXLEN, EMBED_DIM, NUM_HEADS, FF_DIM).to(device)

In [ ]:
# Função de custo e otimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  Diferente da proposta original do artigo Attention Is All You Need (que usa funções estáticas de seno e cosseno), nossa implementação utiliza embeddings posicionais aprendidos no treinamento através de nn.Embedding(maxlen, embed_dim). Qual a vantagem e o possível risco de aprender essas posições do zero em um conjunto de dados pequeno?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
  Na classe IMDBTransformer, aplicamos a função nn.AdaptiveAvgPool1d(1) logo após o bloco codificador. Por que agregamos a dimensão temporal tirando uma média global em vez de simplesmente usarmos um Flatten() (achatamento) antes da camada linear final?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Treinamento do Modelo

A célula abaixo tem a implementação manual do ciclo de treinamento do modelo construido. Diferente do que era feito com o Keras via TensorFlow, no PyTorch vamos fazer um controle explicito de cada etapa da atualização dos pesos. Esse processo é dividido em duas fases para cada época:

* **Fase de Treinamento**: Nessa fase, observe que é utilizado camadas de dropout (`self.dropout1`, `self.dropout2`) para evitar *overfitting*. O comando `model.train()` ativa essas camadas, fazendo com que neurônios aleatórios sejam desligados durante o treino.

Alguns outros detalhes merecem ser detacados:

a. `optimize.zero_grad()`: Limpa os gradientes antigos. No PyTorch, os gradientes se acumulam por padrão. Se não os zerarmos, a atualização dos pesos será errada.

b. `loss.backward()`: Calcula o gradiente da perda em relação a cada parâmetros (*backpropagation*).

c. `optmizer.step()`: Atualiza os pesos do modelo na direção oposta ao gradiente para minimizar o erro.

* **Fase de validação**: Nessa etapa, o modelo avalia dados nunca antes vistos. é chamado o `model.eval()`. Além disso, o dropout é desativado e ajusta a normalização. Para usar toda a capacidadade da rede e fazer uma melhor predição, é chamado `with torch.no_grad()`. Isso desliga os cálculos de gradientes, economizando memória VRAM e processamento.

* **Early Stopping** e salvamento do melhor modelo: Ao final de cada época, a `val_loss` é comparada com o melhor valor obtido até então. Se houve melhora, os pesos do modelo são salvos com `torch.save(model.state_dict(), ...)`, que persiste apenas os parâmetros aprendidos, sem a arquitetura. Se a `val_loss` não melhorar por `PATIENCE` épocas consecutivas, o treino é interrompido antecipadamente para evitar *overfitting*. Ao final, seja por *early stopping* ou por esgotar as épocas, os melhores pesos são restaurados com `model.load_state_dict(torch.load(...))`, garantindo que o modelo usado na avaliação seja o melhor obtido durante o treino, não necessariamente o do último `epoch`.

In [ ]:
# parâmetros de early stopping
PATIENCE = 3  # épocas sem melhora até parar
best_val_loss = float("inf")
epochs_without_improvement = 0
best_model_path = "output/best_model.pt"

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")

    train_acc = 100 * correct / total
    avg_loss = total_loss / len(train_loader)
    print(f"Época [{epoch+1}/{EPOCHS}] Completa. Avg Loss: {avg_loss:.4f}, Acc Treino: {train_acc:.2f}%")

    # avaliação no conjunto de validação
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            val_loss += criterion(outputs, targets).item()  # acumula a loss de validação
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * correct / total
    print(f"Acc Validação: {val_acc:.2f}% | Val Loss: {val_loss:.4f}")

    # salva o melhor modelo e verifica early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)  # salva apenas os pesos
        print(f"novo melhor modelo salvo (val_loss={val_loss:.4f})\n")
    else:
        epochs_without_improvement += 1
        print(f"sem melhora há {epochs_without_improvement}/{PATIENCE} épocas\n")
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping ativado")
            break

# carrega os pesos do melhor checkpoint ao final
model.load_state_dict(torch.load(best_model_path))

## Avaliação do Modelo

Nesta etapa, o modelo treinado é avaliado utilizando textos que não fizeram parte do processo de treinamento, com o objetivo de verificar sua capacidade de generalização. Cada texto passa por um pré-processamento que inclui tokenização, conversão das palavras em índices numéricos com base no vocabulário do IMDB e aplicação de padding para garantir um tamanho fixo de entrada. Em seguida, o modelo realiza a inferência em modo de avaliação, produzindo probabilidades para cada classe de sentimento. A classe com maior probabilidade é selecionada como predição final, juntamente com um valor de confiança, permitindo analisar qualitativamente o desempenho do modelo em novos exemplos.

In [ ]:
def get_imdb_word_index():
    # carrega e ajusta o índice de palavras do imdb para compatibilidade com o treino

     # carrega o dicionário palavra - índice original do IMDB
    word_index = imdb.get_word_index() 
    
    # desloca os índices para reservar tokens especiais
    word_index = {k: (v + 3) for k, v in word_index.items()}
    
    word_index["<PAD>"] = 0 # token usado para padding das sequências
    word_index["<START>"] = 1 # token que indica início da sequência
    word_index["<UNK>"] = 2 # token para palavras desconhecidas
    word_index["<UNUSED>"] = 3 # token reservado e não utilizado
    
    return word_index

In [ ]:
def predict_sentiment(model, texts, word_index, maxlen=500):
    # realiza a predição de sentimento para uma lista de textos

    # coloca o modelo em modo de avaliação (desativa dropout)
    model.eval()  
    sequences = [] # lista para armazenar as sequências tokenizadas
    
    for text in texts:
        # tokeniza o texto e remove pontuações básicas
        words = text.lower().replace('.', '').replace(',', '').replace('!', '').replace('?', '').split()
        
        # converte cada palavra em seu índice correspondente no vocabulário
        sequence = [
            idx if (idx := word_index.get(w, 2)) < VOCAB_SIZE else 2
            for w in words
        ]
        
        # aplica padding ou truncamento para garantir tamanho fixo
        if len(sequence) < maxlen:
            sequence = [0] * (maxlen - len(sequence)) + sequence
        else:
            sequence = sequence[-maxlen:]
            
        sequences.append(sequence) # adiciona a sequência processada à lista
    
    # converte as sequências para tensor e envia para CPU/GPU
    input_tensor = torch.tensor(sequences, dtype=torch.long).to(device)
    
    # executa a inferência sem calcular gradientes
    with torch.no_grad():
        logits = model(input_tensor) # obtém as saídas brutas do modelo
        probabilities = torch.softmax(logits, dim=1) # converte logits em probabilidades
        predictions = torch.argmax(probabilities, dim=1) # seleciona a classe mais provável
        
    results = [] # lista para armazenar os resultados finais
    class_names = ["Negative", "Positive"] # rótulos das classes
    
    for i, text in enumerate(texts):
        pred_idx = predictions[i].item() # índice da classe prevista
        confidence = probabilities[i][pred_idx].item() # confiança da predição

        results.append({
            "text": text,
            "sentiment": class_names[pred_idx],
            "negative_probability": f"{probabilities[i][0].item()*100:.2f}%",
            "positive_probability": f"{probabilities[i][1].item()*100:.2f}%",
            "confidence": f"{confidence*100:.2f}%"
        })
        
    return results


In [ ]:
word_index = get_imdb_word_index()

In [ ]:
my_reviews = [
    "This movie was absolutely fantastic, the acting was great and the plot was amazing.",
    "I hated this film. It was boring, slow, and a complete waste of time.",
    "The visual effects were incredible and the sound design was top tier, but the story was empty and boring.",
    "I really wanted to love this movie. The trailer looked like a masterpiece, but the actual film was a huge letdown.",
    "If you are a fan of mindless action with zero plot, this is perfect. Otherwise, stay away.",
    "Oh, brilliant. Another superhero movie with the exact same plot as the last fifty ones. Just what we needed.",
    "It was not the best movie I have ever seen, but it was certainly not the worst. It was just okay.",
    "I cannot say I didn't enjoy it. It was actually better than I expected.",
    "At first I thought this movie was going to be terrible, but the ending was absolutely perfect.",
    "Funny at times, yet I wouldn't watch it again.",
    "Nice visuals and great performances, but the plot was completely confusing, waste of time and terrible."
]

predictions = predict_sentiment(model, my_reviews, word_index, MAXLEN)

for p in predictions:
    print(f"Análise: '{p['text']}'")
    print(f"Predição: {p['sentiment']} (Probabilidade Positiva: {p['positive_probability']} | Probabilidade Negativa: {p['negative_probability']})")
    print("-" * 50)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 3:</b><br>
  Observe a predição para a resenha "The visual effects were incredible and the sound design was top tier, but the story was empty and boring". Considerando a arquitetura simplificada aplicada (apenas 2 cabeças de atenção e dimensão latente de 32), você acha que o modelo realmente compreendeu a concessão lógica da frase, ou ele simplesmente atribuiu um peso matemático maior à palavra "boring" do que à palavra "incredible"?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

A seguir, vamos inspecionar a matriz de confusão para inspecionar a natureza dos erros cometidos pelo modelo Transformer treinado.

* **Falsos Positivos:** Casos em que o modelo foi "otimista" demais, classificando avaliações negativas como positivas.
* **Falsos Negativos:** Casos em que o modelo foi excessivamente "pessimista", falhando em reconhecer o sentimento positivo de um texto.

A célula a seguir desativa o grafo de computação (`torch.no_grad()`) para garantir eficiência de memória e executa a inferência em todo o conjunto de validação.

In [ ]:
# colocando o modelo em modo de avaliação
model.eval()

# listas para armazenar todas as predições e alvos reais
all_preds = []
all_targets = []

# desativando o cálculo de gradientes para a validação
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        
        # calculando as probabilidades e extraindo as classes previstas
        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(probabilities, dim=1)
        
        # armazenando os resultados convertidos para matrizes numpy
        all_preds.extend(predictions.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# gerando e plotando a matriz de confusão
cm = confusion_matrix(all_targets, all_preds)

# configurando a exibição visual do mapa de calor
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=["Negative", "Positive"], 
    yticklabels=["Negative", "Positive"],
    cbar=False
)
plt.title("Confusion Matrix (Validation Set)", fontsize=14, fontweight='bold', pad=15)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

Após o processamento da sentença pelo bloco Transformer e a aplicação do *Global Average Pooling*, cada texto é matematicamente comprimido em um vetor contextual contínuo de 32 dimensões (o que constitui o espaço latente). 

Como é complexo visualizar hiperplanos de alta dimensionalidade de forma direta, existem técnicas de redução de dimensionalidade que podem ser aplicadas para facilitar essa visualização. Nesta etapa, utilizaremos o t-SNE (*t-Distributed Stochastic Neighbor Embedding*). Ele projeta os dados de 32 para 2 dimensões focando em preservar as estruturas de vizinhança local, mantendo textos semanticamente parecidos próximos uns dos outros.

O gráfico a seguir evidenciará a formação de agrupamentos (clusters) que separam a polaridade dos sentimentos. Contudo, por se tratar de uma arquitetura construída do zero, com baixa capacidade paramétrica (apenas 2 cabeças de atenção) e treinada por poucas épocas, é esperado que exista uma zona de sobreposição. Avaliações com sarcasmo, ironia ou sentimentos mistos tenderão a cair nessa fronteira de ambiguidade.

In [ ]:
# listas para armazenar as características do espaço latente e seus rótulos correspondentes
latent_features = []
true_labels = []

def feature_extraction_hook(module, input, output):
    # função hook para capturar a saída da camada global average pooling, reduzindo a última dimensão do formato (batch_size, embed_dim, 1)
    
    squeezed_output = output.squeeze(-1)
    latent_features.append(squeezed_output.detach().cpu().numpy())

# hook no modelo existente sem modificar sua classe de origem
hook_handle = model.global_avg_pool.register_forward_hook(feature_extraction_hook)

# garantindo que o modelo esteja em modo de avaliação para desativar o dropout
model.eval()

# limitando a visualização a um subconjunto (ex: 1000) para evitar tempo excessivo no cálculo do t-sne
num_samples_to_visualize = 1000
collected_samples = 0

with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(device)
        
        # o forward pass aciona automaticamente o hook e preenche a lista latent_features
        _ = model(inputs) 
        
        true_labels.extend(targets.cpu().numpy())
        collected_samples += inputs.size(0)
        
        if collected_samples >= num_samples_to_visualize:
            break

# removendo o hook com segurança após a extração para evitar vazamentos de memória
hook_handle.remove()

# empilhando as listas em arrays numpy para o scikit-learn
latent_matrix = np.vstack(latent_features)[:num_samples_to_visualize]
labels_array = np.array(true_labels)[:num_samples_to_visualize]

print(f"Shape da matriz latente: {latent_matrix.shape}")

# aplicando t-sne para redução de dimensionalidade não linear
tsne = TSNE(n_components=2, random_state=42, perplexity=30.0, max_iter=1000)
tsne_results = tsne.fit_transform(latent_matrix)

# configurando o estilo visual para uma saída profissional
sns.set_theme(style="whitegrid", palette="muted")
plt.figure(figsize=(10, 8))

# criando o gráfico de dispersão
scatter_plot = sns.scatterplot(
    x=tsne_results[:, 0],
    y=tsne_results[:, 1],
    hue=labels_array,
    palette=["#e74c3c", "#3498db"], # vermelho para negativo, azul para positivo
    alpha=0.75,
    edgecolor="k",
    s=60
)

# customizando títulos e rótulos
plt.title("Visualização t-SNE do Espaço Latente (Transformer)", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Componente t-SNE 1", fontsize=12)
plt.ylabel("Componente t-SNE 2", fontsize=12)

# customizando a legenda
legend_handles, _ = scatter_plot.get_legend_handles_labels()
plt.legend(
    handles=legend_handles, 
    labels=["Negative (0)", "Positive (1)"], 
    title="Sentimento",
    loc="best",
    fontsize=11
)

plt.tight_layout()
plt.show()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 4:</b><br>
  No gráfico do espaço latente, notamos dois agrupamentos polarizados, mas também uma inevitável zona de sobreposição (pontos vermelhos e azuis misturados no centro). Se você pudesse ler os textos exatos que caíram nessa fronteira de ambiguidade, que tipo de estrutura linguística ou figura de linguagem você esperaria encontrar?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>